# Brain · use 1 · healthy structure segmentation*A small model that names the structures of a healthy brain on T1 MRI, and is honestabout what it gets wrong.*> **For learning anatomy, never for diagnosis.** This notebook trains on labelled public> MRI from [Mindboggle-101](DATA.md) and shows its output beside the dataset's own> labels. It makes no clinical claim, and nothing here may be used to say what a person> has.> **This is a demonstrator.** It trains a very small model, on a few subjects, on a few> 2D slices, in about a minute on a laptop CPU. It is meant to be read end to end in one> sitting. The last section says exactly which numbers to change for a full run, and what> that would cost.**What you will see, in order**1. the healthy anatomy this use is about;2. the data, and where each label comes from;3. a small segmentation model;4. an evaluation that splits by subject;5. the errors, looked at rather than hidden.

## 1. The healthy anatomyA T1-weighted MRI is the sequence anatomy is usually taught on: it separates the tissuesof the brain well. This use asks a model to find, on that image, the structures a studentlearns to recognise first:| Structure | Where it is ||---|---|| **Cerebral cortex** | the folded grey matter at the surface of the hemispheres || **Cerebral white matter** | beneath the cortex, the fibre tissue that connects it || **Lateral ventricles** | the paired CSF spaces at the centre of the hemispheres || **Thalamus** | the deep grey nucleus on each side of the third ventricle || **Hippocampus** | in the medial temporal lobe, along the floor of the temporal horn |The healthy brain comes first in this atlas because every lesion is read against it: tosee that something has displaced the ventricle, you must first know the ventricle.

## 2. The data, and where each label comes fromThe dataset is Mindboggle-101: 101 healthy adults, each with a T1 image and two labelvolumes. [`DATA.md`](DATA.md) records how it is fetched, its checksums, and its terms.The data is **not** in this repository; it sits under `data/`, which git ignores.**One thing matters more than any other here.** The two label volumes do not have thesame provenance:- `labels.DKT31.manual.nii.gz` — the **cortex**, labelled by hand following the DKT  protocol. These are expert labels.- `labels.DKT31.manual+aseg.nii.gz` — the same cortical labels, **plus** the non-cortical  structures taken from FreeSurfer's automatic segmentation (*aseg*). These are a  program's output, not an expert's.So a model trained on the cortex learns from people, and a model trained on theventricles, thalamus or hippocampus learns to **imitate FreeSurfer**, including whereverFreeSurfer is wrong. This notebook keeps the two apart, marks every structure with itsprovenance, and repeats it next to the results.

In [ ]:
from pathlib import Path

# --- what the demonstrator runs on. Raise these for a real run (see the last section).
CONFIG = dict(
    subjects_per_cohort_train=3,   # 5 cohorts -> 15 training subjects
    subjects_per_cohort_test=2,    #           -> 10 held-out subjects
    slices_per_subject=24,         # axial slices, sampled through the labelled range
    image_size=96,                 # 192 mm field of view, downsampled by 2
    epochs=25,
    batch_size=8,
    learning_rate=3e-3,
    base_channels=8,               # width of the small U-Net
    seed=0,
)

DATA = Path("data/extracted/Mindboggle101_release3/Mindboggle101_volumes")
assert DATA.is_dir(), f"data not found at {DATA} - see DATA.md for how to fetch it"
print("data root:", DATA)

In [ ]:
import numpy as np, torch

np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.use_deterministic_algorithms(True, warn_only=True)
print("numpy", np.__version__, "| torch", torch.__version__, "| threads", torch.get_num_threads())

### 2.1 The subjects, and the splitThe split is **by subject**, never by slice. Two slices of the same brain look almostalike; if one went to training and the other to the test set, the score would measurememory, not generalisation. Subjects are also drawn from every cohort, so that trainingand test both span the five acquisition sources.

In [ ]:
cohorts = {d.name.replace("_volumes", ""): sorted(p for p in d.iterdir() if p.is_dir())
           for d in sorted(DATA.iterdir()) if d.is_dir()}

train_subjects, test_subjects = [], []
for name, subs in cohorts.items():
    n_tr, n_te = CONFIG["subjects_per_cohort_train"], CONFIG["subjects_per_cohort_test"]
    train_subjects += subs[:n_tr]
    test_subjects += subs[n_tr:n_tr + n_te]

print(f"{sum(len(s) for s in cohorts.values())} subjects available, by cohort:")
for name, subs in cohorts.items():
    print(f"  {name:14s} {len(subs):3d}")
print(f"\ntraining on {len(train_subjects)} subjects, holding out {len(test_subjects)}")
assert not (set(train_subjects) & set(test_subjects)), "a subject is in both sets"
print("no subject is in both sets")

### 2.2 The classes, and their provenanceFive structures plus background. The provenance column is not decoration: it is thedifference between a label a person drew and a label a program produced. The numericcodes of the non-cortical structures follow FreeSurfer's colour table, as recorded in[`DATA.md`](DATA.md).

In [ ]:
CLASSES = [
    # index, name,                     provenance,              source codes
    (0, "background",                  "-",                     ()),
    (1, "cerebral cortex",             "manual (DKT protocol)", "labels.DKT31.manual >= 1000"),
    (2, "cerebral white matter",       "automatic (FreeSurfer aseg)", (2, 41)),
    (3, "lateral ventricle",           "automatic (FreeSurfer aseg)", (4, 43)),
    (4, "thalamus",                    "automatic (FreeSurfer aseg)", (10, 49)),
    (5, "hippocampus",                 "automatic (FreeSurfer aseg)", (17, 53)),
]
N_CLASSES = len(CLASSES)
MANUAL = {1}            # the only class a human drew

for i, name, prov, codes in CLASSES:
    print(f"{i}  {name:24s} {prov:30s} {codes}")

In [ ]:
import nibabel as nib

def build_labels(aseg, manual):
    """Map the dataset's label codes onto the six teaching classes."""
    y = np.zeros(aseg.shape, np.uint8)
    for idx, _, _, codes in CLASSES:
        if isinstance(codes, tuple) and codes:
            for c in codes:
                y[aseg == c] = idx
    y[manual >= 1000] = 1          # manual cortex wins where the two overlap
    return y


def load_subject(subject, n_slices, size):
    """Return (slices, labels) as arrays of shape (n, size, size)."""
    img = nib.as_closest_canonical(nib.load(subject / "t1weighted_brain.nii.gz"))
    aseg = np.asarray(nib.as_closest_canonical(
        nib.load(subject / "labels.DKT31.manual+aseg.nii.gz")).dataobj).astype(np.int32)
    manual = np.asarray(nib.as_closest_canonical(
        nib.load(subject / "labels.DKT31.manual.nii.gz")).dataobj).astype(np.int32)
    vol = img.get_fdata(dtype=np.float32)

    y = build_labels(aseg, manual)
    lo, hi = np.percentile(vol[vol > 0], (1, 99))       # per subject, so scanners differ less
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    keep = [k for k in range(y.shape[2]) if (y[:, :, k] > 0).sum() > 4000]
    keep = keep[:: max(1, len(keep) // n_slices)][:n_slices]

    half = size                                         # crop 2*size mm, then take every 2nd voxel
    cx, cy = vol.shape[0] // 2, vol.shape[1] // 2
    X, Y = [], []
    for k in keep:
        a = vol[max(0, cx - half):cx + half, max(0, cy - half):cy + half, k]
        b = y[max(0, cx - half):cx + half, max(0, cy - half):cy + half, k]
        pad = ((0, max(0, 2 * half - a.shape[0])), (0, max(0, 2 * half - a.shape[1])))
        X.append(np.pad(a, pad)[::2, ::2])
        Y.append(np.pad(b, pad)[::2, ::2])
    return np.stack(X).astype(np.float32), np.stack(Y).astype(np.int64)


def load_set(subjects):
    xs, ys = zip(*(load_subject(s, CONFIG["slices_per_subject"], CONFIG["image_size"])
                   for s in subjects))
    return np.concatenate(xs), np.concatenate(ys)

In [ ]:
import time

t0 = time.time()
X_train, Y_train = load_set(train_subjects)
print(f"{len(X_train)} training slices of {X_train.shape[1]}x{X_train.shape[2]}"
      f"  ({time.time() - t0:.1f}s)")

share = np.bincount(Y_train.reshape(-1), minlength=N_CLASSES) / Y_train.size
print("\nhow much of the image each class occupies:")
for (i, name, prov, _), s in zip(CLASSES, share):
    print(f"  {name:24s} {100 * s:5.2f}%   {prov}")

The imbalance is the whole difficulty of this task. The hippocampus is a few tenths ofa percent of the image; a model that simply never predicted it would still be rightalmost everywhere. Two things answer that below: a loss that weights the rare classes up,and a metric (Dice) that does not reward predicting nothing.

## 3. A small modelA small U-Net: three levels, eight channels at the top, about thirty thousandparameters. That is deliberate — the atlas ships one small model per organ, readable andretrainable on a laptop, rather than one large model for the whole body.

In [ ]:
import torch.nn as nn

class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class TinyUNet(nn.Module):
    def __init__(self, n_classes=N_CLASSES, base=CONFIG["base_channels"]):
        super().__init__()
        b = base
        self.enc1, self.enc2, self.enc3 = Block(1, b), Block(b, 2 * b), Block(2 * b, 4 * b)
        self.pool = nn.MaxPool2d(2)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, 2), Block(4 * b, 2 * b)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * b, b, 2, 2), Block(2 * b, b)
        self.head = nn.Conv2d(b, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(e3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)


model = TinyUNet()
print(f"{sum(p.numel() for p in model.parameters()):,d} parameters")

## 4. TrainingCross-entropy, weighted by the inverse square root of each class's share, so thehippocampus is not drowned by the background. The demonstrator runs a fixed number ofepochs on the sampled slices; nothing is tuned on the held-out subjects.

In [ ]:
weights = 1.0 / np.sqrt(share)
weights = torch.tensor(weights / weights.mean(), dtype=torch.float32)
print("loss weights:", " ".join(f"{name.split()[0]}={w:.2f}"
                                for (_, name, _, _), w in zip(CLASSES, weights)))

optimiser = torch.optim.Adam(model.parameters(), CONFIG["learning_rate"])
criterion = nn.CrossEntropyLoss(weight=weights)

Xt = torch.from_numpy(X_train)[:, None]
Yt = torch.from_numpy(Y_train)

t0 = time.time()
history = []
model.train()
for epoch in range(CONFIG["epochs"]):
    order = torch.randperm(len(Xt))
    total = 0.0
    for i in range(0, len(Xt), CONFIG["batch_size"]):
        idx = order[i:i + CONFIG["batch_size"]]
        optimiser.zero_grad()
        loss = criterion(model(Xt[idx]), Yt[idx])
        loss.backward()
        optimiser.step()
        total += loss.item() * len(idx)
    history.append(total / len(Xt))
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"  epoch {epoch + 1:3d}   loss {history[-1]:.4f}   ({time.time() - t0:.0f}s)")
print(f"trained in {time.time() - t0:.0f}s on CPU")

## 5. Evaluation: the optimised runThe numbers that count for this use do not come from the training above. They come from aseparate run of [`colab_train.py`](colab_train.py) on a GPU: the same data and the samesplit rule, but every subject, every labelled slice at full resolution, eleven structuresand sixty epochs. That run is recorded in [`results_t4.json`](results_t4.json), which thisnotebook reads rather than restates, so the notebook, the figure and the model card cannever disagree.Read the provenance with every score. The cortex is measured against labels a persondrew. Every other row is measured against FreeSurfer's automatic output: a high scorethere means **agrees with FreeSurfer**, not **correct**.

In [ ]:
import json

RESULTS = json.loads(Path("results_t4.json").read_text())
run, cfg_run, split = RESULTS["run"], RESULTS["config"], RESULTS["split"]

print(f"run of {run['script']}  seed {run['seed']}  on {run['environment']['device']}")
print(f"  {cfg_run['parameters']:,d} parameters, {cfg_run['depth']} levels, "
      f"{cfg_run['base_channels']} channels, {cfg_run['input_slices']} input slices, "
      f"{cfg_run['crop']}x{cfg_run['crop']}")
print(f"  {split['train_subjects']} train / {split['val_subjects']} val / "
      f"{split['held_out_subjects']} held out, split {split['by']}")
print(f"  {split['slices']['train']:,d} training slices, "
      f"{split['slices']['held_out']:,d} held-out slices")
print(f"  best epoch {run['training']['best_epoch']} of {run['training']['epochs']}, "
      f"validation Dice {run['training']['best_val_mean_dice']:.4f}, "
      f"{run['training']['seconds'] / 60:.0f} min of GPU")
print(f"  data: {RESULTS['dataset']['name']} {RESULTS['dataset']['doi']}, "
      f"{RESULTS['dataset']['files_verified']} files verified by MD5")

In [ ]:
for s in sorted(RESULTS["structures"], key=lambda s: -s["mean_dice"]):
    drawn = "  <- drawn by a person" if s["provenance"].startswith("manual") else ""
    print(f"{s['name']:24s} {s['mean_dice']:6.3f}   {s['provenance']}{drawn}")

print()
for label, group in RESULTS["by_provenance"].items():
    if isinstance(group, dict):
        print(f"{label:30s} {group['structures']:2d} structure(s), "
              f"mean of means {group['mean_of_structure_means']:.4f}")
print(RESULTS["by_provenance"]["read_this_first"])

In [ ]:
table = RESULTS["per_subject_dice"]["rows"]
short = ["cortex", "white m.", "ventric.", "thalam.", "caudate", "putamen",
         "pallid.", "hippoc.", "amygd.", "brainst.", "cerebel."]

print(f"{'held-out subject':17s} | " + " | ".join(f"{s:>8s}" for s in short))
for name, values in table.items():
    print(f"{name:17s} | " + " | ".join(f"{v:8.3f}" for v in values))

means = {name: sum(v) / len(v) for name, v in table.items()}
ranked = sorted(means.items(), key=lambda kv: kv[1])
print(f"\nworst subject: {ranked[0][0]} at {ranked[0][1]:.3f}, "
      f"next worst {ranked[1][0]} at {ranked[1][1]:.3f}")
cortex_wo = [v[0] for n, v in table.items() if n != ranked[0][0]]
print(f"cortex without {ranked[0][0]}: {sum(cortex_wo) / len(cortex_wo):.3f} "
      f"(min {min(cortex_wo):.3f}, max {max(cortex_wo):.3f})")

![Dice per structure on fifteen held-out brains, sorted, each subject a dot, with the provenance of every label shown](../../../docs/figures/png/brain/03_use1_results.png)The figure sorts the structures by score and draws every held-out subject as a dot, sothe spread shows rather than being averaged away.

### The demonstrator's own numbersFor completeness, what the small model trained above scores on its own held-outsubjects. It is a check that the pipeline works, not a result: fewer subjects, fewerslices, six structures, a fraction of the parameters.

In [ ]:
def dice(pred, truth, cls):
    p, g = pred == cls, truth == cls
    denom = p.sum() + g.sum()
    return np.nan if denom == 0 else float(2 * (p & g).sum() / denom)


model.eval()
demo_rows, names = [], []
with torch.no_grad():
    for subject in test_subjects:
        x, y = load_subject(subject, CONFIG["slices_per_subject"], CONFIG["image_size"])
        pred = model(torch.from_numpy(x)[:, None]).argmax(1).numpy()
        demo_rows.append([dice(pred, y, c) for c in range(1, N_CLASSES)])
        names.append(subject.name)

demo_scores = np.array(demo_rows)
print(f"{'demonstrator':16s} | " + " | ".join(f"{n.split()[-1][:9]:>9s}" for _, n, _, _ in CLASSES[1:]))
for name, row in zip(names, demo_scores):
    print(f"{name:16s} | " + " | ".join(f"{v:9.3f}" for v in row))
print(f"{'mean':16s} | " + " | ".join(f"{v:9.3f}" for v in np.nanmean(demo_scores, 0)))

## 6. The errors, looked at**One brain behaves unlike the others.** In the optimised run, `Colin27-1` averages 0.459where the next worst subject averages 0.853. That is not a scanning accident: Colin27 isa *template*, built by averaging many scans of one person, so its contrast and its edgeslook like no individual brain. The model was trained on individuals and is being askedabout an average. It is kept in the table rather than quietly dropped, because it showswhat happens at the edge of the distribution a model was trained on. Without it, thecortex averages 0.966 instead of 0.928.**Small and deep is hard.** The pallidum, the smallest of the deep grey nuclei, comeslast at 0.758, and it is where a single subject falls furthest: `HLN-12-1` scores 0.390on it while scoring above 0.85 almost everywhere else. The cerebellum and the brainstem,large and well separated, sit above 0.93. That ordering is the lesson, not the headlinenumber.The cells below show, on the notebook's own demonstrator, where a segmentation model'sdisagreement actually lives: on the boundaries.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

worst_demo = int(np.nanmean(demo_scores, 1).argmin())
subject = test_subjects[worst_demo]
x, y = load_subject(subject, CONFIG["slices_per_subject"], CONFIG["image_size"])
with torch.no_grad():
    pred = model(torch.from_numpy(x)[:, None]).argmax(1).numpy()

k = int(np.argmax([(y[i] > 0).sum() for i in range(len(y))]))
cmap = ListedColormap(["#0d0d0c", "#3987e5", "#6f6e66", "#c3c2b7", "#d95926", "#f8f7f4"])

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(x[k].T, cmap="gray", origin="lower"); axes[0].set_title("T1 slice")
axes[1].imshow(y[k].T, cmap=cmap, vmin=0, vmax=5, origin="lower")
axes[1].set_title("dataset labels: cortex manual, rest FreeSurfer")
axes[2].imshow(pred[k].T, cmap=cmap, vmin=0, vmax=5, origin="lower")
axes[2].set_title("demonstrator output")
for a in axes:
    a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"the notebook's demonstrator on {subject.name} - not the optimised run")
plt.tight_layout()
plt.show()

In [ ]:
# where the disagreement sits: boundary voxels against interior voxels
def interior_of(mask):
    """Erode a 2D mask by one voxel, with numpy alone (4-neighbour)."""
    up, down = np.roll(mask, 1, 0), np.roll(mask, -1, 0)
    left, right = np.roll(mask, 1, 1), np.roll(mask, -1, 1)
    inner = mask & up & down & left & right
    inner[0, :] = inner[-1, :] = inner[:, 0] = inner[:, -1] = False
    return inner


boundary_hits = interior_hits = boundary_total = interior_total = 0
for cls in range(1, N_CLASSES):
    g = y == cls
    interior = np.stack([interior_of(g[i]) for i in range(len(g))])
    boundary = g & ~interior
    boundary_hits += int((pred[boundary] == cls).sum()); boundary_total += int(boundary.sum())
    interior_hits += int((pred[interior] == cls).sum()); interior_total += int(interior.sum())

print("on this subject, the demonstrator labels correctly:")
print(f"  {100 * interior_hits / max(interior_total, 1):5.1f}% of interior voxels")
print(f"  {100 * boundary_hits / max(boundary_total, 1):5.1f}% of boundary voxels")
print("\nthe disagreement lives on the boundaries, as it should for a segmentation model")

## 7. What this is, and what it is not**Two runs live in this folder, and they are not the same thing.**| | This notebook | The optimised run ||---|---|---|| purpose | read the pipeline end to end in a minute | the numbers of record for this use || subjects | 15, capped by `CONFIG` | 71 train, 15 val, 15 held out || slices | 24 per subject, 96x96 | every labelled slice, 192x192 || structures | 6 | 11 || model | about 30,000 parameters | 1,942,764 parameters || hardware | a laptop CPU, about a minute | a Tesla T4, about 35 minutes || recorded in | nothing: it is a demonstration | `results_t4.json`, `MODEL_CARD.md` |To reproduce the optimised run: `python colab_train.py` on a GPU runtime. The seed andthe configuration are in the script, and it fetches the data itself, checking all eightMD5 sums against the record before it starts.**What neither run is.** Not a validated segmentation tool, not a statement about anyindividual, and not something to decide anything about a person with. The Dice valuesdescribe sampled axial slices of healthy brains from one dataset. Where the labels comefrom FreeSurfer, the model inherits FreeSurfer's mistakes, and a high score there onlymeans the two agree.**The weights.** They are not published. The T1 images carry the terms of the projectsthat acquired them (OASIS, NKI, MMRR), which conflict with the record's CC BY 4.0; underthe project rule the weights are held back until those owners confirm. They areregenerable from the script and the seed. See [`DATA.md`](DATA.md), the[model card](MODEL_CARD.md) and the[licence rule](../../../../CATALOGUE.md#two-licences-kept-apart).**Notebook outputs are not committed.** Rendering a slice puts a piece of the MRI in thefile. The data stays with its provider: run the notebook to see the images.